# 02 Rebuild dataset from raw BDD100K zip files

This notebook rebuilds the current project dataset directly from the three raw archives in `EcoCAR/downloads` without relying on the old notebook2 pipeline.

In [ ]:
from pathlib import Path
import json, os, sys

PROJECT_ROOT = Path('/content/drive/MyDrive/EcoCAR/DETR_GeoLane_pipeline')
if not PROJECT_ROOT.exists():
    PROJECT_ROOT = Path('/content/DETR_GeoLane_pipeline')
assert PROJECT_ROOT.exists(), f'Project root not found: {PROJECT_ROOT}'
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

ECOCAR_ROOT = Path('/content/drive/MyDrive/EcoCAR')
DOWNLOADS = ECOCAR_ROOT / 'downloads'
DATASETS_DRIVE = ECOCAR_ROOT / 'datasets'
RAW_ROOT = Path('/content/bdd100k_raw')
OUTPUT_ROOT = Path('/content/bdd100k_vehicle5')
OUTPUT_ROOT_DRIVE = DATASETS_DRIVE / OUTPUT_ROOT.name
ARCHIVE_PATH_DRIVE = DATASETS_DRIVE / f'{OUTPUT_ROOT.name}_nb02.tar'

print('PROJECT_ROOT       =', PROJECT_ROOT)
print('DOWNLOADS          =', DOWNLOADS)
print('DATASETS_DRIVE     =', DATASETS_DRIVE)
print('RAW_ROOT           =', RAW_ROOT)
print('OUTPUT_ROOT(local) =', OUTPUT_ROOT)
print('OUTPUT_ROOT(drive) =', OUTPUT_ROOT_DRIVE)
print('ARCHIVE_PATH_DRIVE =', ARCHIVE_PATH_DRIVE)



In [ ]:
from src.data_prep import inspect_download_archives

archive_preview = inspect_download_archives(DOWNLOADS)
for name, members in archive_preview.items():
    print('\n' + '='*90)
    print(name)
    if not members:
        print('NOT FOUND')
    else:
        for m in members[:30]:
            print(m)

In [ ]:
from src.data_prep import rebuild_dualpath_dataset, persist_dataset_artifacts

summary = rebuild_dualpath_dataset(
    downloads_dir=DOWNLOADS,
    raw_root=RAW_ROOT,
    output_root=OUTPUT_ROOT,
    force_reextract=False,
)

persist_summary = persist_dataset_artifacts(
    local_dataset_root=OUTPUT_ROOT,
    drive_datasets_dir=DATASETS_DRIVE,
    archive_name=ARCHIVE_PATH_DRIVE.name,
)

summary['drive_dataset_root'] = persist_summary['drive_dataset_root']
summary['archive_path'] = persist_summary['archive_path']
summary['copied_files'] = persist_summary['copied_files']

print(json.dumps(summary, indent=2))



In [ ]:
from collections import Counter

def scan_raw_ids(label_dir, max_files=2000):
    counter = Counter()
    txts = sorted(label_dir.glob('*.txt'))[:max_files]
    for p in txts:
        for line in p.read_text().splitlines():
            s = line.strip().split()
            if len(s) >= 5:
                counter[int(float(s[0]))] += 1
    return counter

train_counter = scan_raw_ids(OUTPUT_ROOT / 'labels' / 'train', 3000)
val_counter = scan_raw_ids(OUTPUT_ROOT / 'labels' / 'val', 1000)
print('train raw ids ->', dict(sorted(train_counter.items())))
print('val raw ids   ->', dict(sorted(val_counter.items())))
print('expected ids  -> {0: car, 1: truck, 2: bus, 3: motorcycle, 4: bicycle}')

In [ ]:
yaml_path = OUTPUT_ROOT / 'bdd100k_vehicle5.yaml'
paths_cfg = OUTPUT_ROOT / 'paths_config.yaml'
print('Dataset YAML:', yaml_path)
print(yaml_path.read_text())
print('Paths config:', paths_cfg)
print(paths_cfg.read_text())

In [ ]:
# Verify archive + Drive artifacts
from pathlib import Path

print('Drive dataset root exists:', OUTPUT_ROOT_DRIVE.exists(), OUTPUT_ROOT_DRIVE)
print('Archive exists:', ARCHIVE_PATH_DRIVE.exists(), ARCHIVE_PATH_DRIVE)
if ARCHIVE_PATH_DRIVE.exists():
    print('Archive size (GB):', round(ARCHIVE_PATH_DRIVE.stat().st_size / 1024**3, 3))
for p in [OUTPUT_ROOT_DRIVE / 'bdd100k_vehicle5.yaml', OUTPUT_ROOT_DRIVE / 'paths_config.yaml']:
    print(p, 'exists=', p.exists())
